# Task 2 Candidate Reranker Pipeline

This notebook tests a new design:

```text
LongProfile -> Task 1 parent-industry candidates
SegmentName + SegmentDescription + candidate taxonomy -> Task 2 match score
rank candidate SubIndustries -> top 5 predictions
```

The Task 2 cross-encoder does **not** receive `LongProfile` directly. LongProfile is only used by the Task 1 candidate generator.

## Run Modes

Default flags are safe: they load/validate data, build taxonomy, and define functions.

Recommended experiment order:

1. Set `RUN_TASK1_BASELINE=True` to train the LongProfile -> parent candidate model.
2. Set `RUN_BUILD_PAIR_DATA=True` to create Task 2 segment-taxonomy training pairs.
3. Set `RUN_CROSS_ENCODER_TRAINING=True` to train the pairwise reranker.
4. Set `RUN_FULL_PIPELINE_EVAL=True` after both models exist.

In [53]:
import html
import json
import math
import os
import random
import re
import time
import warnings
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("default")

# Avoid Windows CUDA/NVRTC JIT-fusion issues seen with DeBERTa relative-position ops.
for _jit_name, _jit_args in [
    ("_jit_set_nvfuser_enabled", (False,)),
    ("_jit_set_texpr_fuser_enabled", (False,)),
    ("_jit_override_can_fuse_on_gpu", (False,)),
]:
    try:
        getattr(torch._C, _jit_name)(*_jit_args)
    except Exception:
        pass

In [54]:
TASK2_PATH = Path("task2_subindustry_classification_final.csv")
TASK1_PATH = Path("task1_gecs_classification_final.csv")
GECS_PATH = Path("GECS_Activities2026.xlsx")
OUTPUT_DIR = Path("models/task2_candidate_reranker")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK1_BASELINE_PATH = OUTPUT_DIR / "task1_parent_tfidf_sgd.joblib"
PAIR_TRAIN_PATH = OUTPUT_DIR / "task2_cross_encoder_train_pairs.csv"
PAIR_VAL_PATH = OUTPUT_DIR / "task2_cross_encoder_val_pairs.csv"
PAIR_TEST_PATH = OUTPUT_DIR / "task2_cross_encoder_test_pairs.csv"

MODEL_NAME = "microsoft/deberta-v3-small"
LOCAL_FILES_ONLY = True
FORCE_ENCODER_FLOAT32 = True

RUN_TASK1_BASELINE = False
RUN_BUILD_PAIR_DATA = False
RUN_CROSS_ENCODER_TRAINING = False
RUN_FULL_PIPELINE_EVAL = True
RUN_SMOKE_FORWARD = False

SEED = 42
MIN_EVAL_SUPPORT = 5
TASK1_TOP_K = 5
LOW_CONFIDENCE_THRESHOLD = 0.50
FALLBACK_PARENT_TOP_K = 10
MIN_CANDIDATES = 20

NEG_SAME_PARENT = 3
NEG_SAME_GROUP = 3
NEG_SEMANTIC = 3
NEG_RANDOM = 3
DEBUG_PAIR_ROWS = None  # set to e.g. 1000 for a quick dry run

MAX_LENGTH = 256
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
GRAD_CLIP = 1.0
EARLY_STOP_PATIENCE = 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything()
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR.resolve())

Device: cuda
Output directory: C:\Users\shami\OneDrive\Desktop\capstone\models\task2_candidate_reranker


## Text and Code Helpers

In [55]:
SPECIAL_CHAR_TRANSLATION = str.maketrans({
    "\u2018": "'",
    "\u2019": "'",
    "\u201c": '"',
    "\u201d": '"',
    "\u2013": "-",
    "\u2014": "-",
    "\u2212": "-",
    "\xa0": " ",
    "\ufeff": "",
})

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = text.translate(SPECIAL_CHAR_TRANSLATION)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_code(value, width=None):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    text = re.sub(r"\D+", "", text)
    if width is not None and text:
        text = text.zfill(width)
    return text

def build_segment_text(segment_name, segment_description):
    return (
        f"[SEGMENT_NAME] {normalize_text(segment_name)} "
        f"[SEGMENT_DESCRIPTION] {normalize_text(segment_description)}"
    ).strip()

def build_profile_text(long_profile):
    return f"[COMPANY_PROFILE] {normalize_text(long_profile)}".strip()

## Read GECS Taxonomy

In [56]:
XLSX_NS = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

def _column_to_index(cell_ref):
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    idx = 0
    for ch in letters:
        idx = idx * 26 + ord(ch.upper()) - 64
    return idx - 1

def _read_shared_strings(zf):
    if "xl/sharedStrings.xml" not in zf.namelist():
        return []
    root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
    shared = []
    for item in root.findall("a:si", XLSX_NS):
        shared.append("".join(t.text or "" for t in item.findall(".//a:t", XLSX_NS)))
    return shared

def read_xlsx_sheet_xml(path, sheet_index=0):
    with zipfile.ZipFile(path) as zf:
        shared = _read_shared_strings(zf)
        sheet_paths = [
            name for name in zf.namelist()
            if re.fullmatch(r"xl/worksheets/sheet\d+\.xml", name)
        ]
        sheet_paths = sorted(
            sheet_paths,
            key=lambda name: int(re.search(r"sheet(\d+)\.xml", name).group(1)),
        )
        root = ET.fromstring(zf.read(sheet_paths[sheet_index]))
        rows = []
        for row in root.findall(".//a:sheetData/a:row", XLSX_NS):
            values = {}
            for cell in row.findall("a:c", XLSX_NS):
                col_idx = _column_to_index(cell.attrib.get("r", "A"))
                value = ""
                if cell.attrib.get("t") == "inlineStr":
                    value = "".join(t.text or "" for t in cell.findall(".//a:t", XLSX_NS))
                else:
                    v = cell.find("a:v", XLSX_NS)
                    if v is not None:
                        value = v.text or ""
                        if cell.attrib.get("t") == "s" and value != "":
                            value = shared[int(value)]
                values[col_idx] = value
            if values:
                rows.append([values.get(i, "") for i in range(max(values) + 1)])
    header = [normalize_text(x) for x in rows[0]]
    body = []
    for row in rows[1:]:
        padded = row + [""] * (len(header) - len(row))
        body.append(padded[:len(header)])
    return pd.DataFrame(body, columns=header)

def read_gecs_workbook(path):
    try:
        return pd.read_excel(path, sheet_name=0, dtype=str)
    except Exception as exc:
        print(f"pandas.read_excel fallback: {type(exc).__name__}: {exc}")
        return read_xlsx_sheet_xml(path, sheet_index=0)

def pick_column(frame, candidates):
    normalized = {re.sub(r"[^a-z0-9]+", "", str(c).lower()): c for c in frame.columns}
    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]+", "", candidate.lower())
        if key in normalized:
            return normalized[key]
    raise KeyError(f"None of these columns found: {candidates}. Available: {frame.columns.tolist()}")

assert GECS_PATH.exists(), f"Missing GECS file: {GECS_PATH}"
raw_gecs = read_gecs_workbook(GECS_PATH)

col_leaf = pick_column(raw_gecs, ["Activity ID", "ActivityID"])
col_activity = pick_column(raw_gecs, ["Activity"])
col_def = pick_column(raw_gecs, ["Activity Definition", "Definition"])
col_parent = pick_column(raw_gecs, ["Industry ID", "IndustryID"])
col_industry = pick_column(raw_gecs, ["Industry Name (current)", "Industry", "Industry Name"])
col_group = pick_column(raw_gecs, ["Industry Group ID", "IndustryGroupID"])
col_group_name = pick_column(raw_gecs, ["Industry Group Name (current)", "Industry Group Name", "Industry Group"])
col_sector = pick_column(raw_gecs, ["Sector ID", "SectorID"])
col_sector_name = pick_column(raw_gecs, ["Sector Name (current)", "Sector Name", "Sector"])

taxonomy = raw_gecs[[col_sector, col_sector_name, col_group, col_group_name, col_parent, col_industry, col_leaf, col_activity, col_def]].copy()
taxonomy.columns = [
    "sector_code",
    "sector_name",
    "industry_group_code",
    "industry_group_name",
    "industry_code",
    "industry_name",
    "subindustry_code",
    "subindustry_name",
    "subindustry_description",
]

taxonomy["sector_code"] = taxonomy["sector_code"].map(lambda x: clean_code(x, width=3))
taxonomy["industry_group_code"] = taxonomy["industry_group_code"].map(lambda x: clean_code(x, width=5))
taxonomy["industry_code"] = taxonomy["industry_code"].map(lambda x: clean_code(x, width=8))
taxonomy["subindustry_code"] = taxonomy["subindustry_code"].map(lambda x: clean_code(x, width=10))
for col in ["sector_name", "industry_group_name", "industry_name", "subindustry_name", "subindustry_description"]:
    taxonomy[col] = taxonomy[col].map(normalize_text)

taxonomy = taxonomy[taxonomy["subindustry_code"].str.len() == 10].drop_duplicates("subindustry_code").reset_index(drop=True)

taxonomy["taxonomy_text"] = (
    "[SECTOR] " + taxonomy["sector_name"].fillna("") + " "
    "[INDUSTRY_GROUP] " + taxonomy["industry_group_name"].fillna("") + " "
    "[INDUSTRY] " + taxonomy["industry_name"].fillna("") + " "
    "[SUBINDUSTRY] " + taxonomy["subindustry_name"].fillna("") + " "
    "[DEFINITION] " + taxonomy["subindustry_description"].fillna("")
).map(normalize_text)

taxonomy.to_csv(OUTPUT_DIR / "taxonomy_table.csv", index=False)
display(taxonomy.head())
print("Taxonomy subindustries:", taxonomy["subindustry_code"].nunique())

,sector_code,sector_name,industry_group_code,industry_group_name,industry_code,industry_name,subindustry_code,subindustry_name,subindustry_description,taxonomy_text
0,101,Basic Materials,10110,Agriculture,10110010,Agricultural Inputs,1011001001,Pesticide and Agrichemical Manufacturer,Companies engaged in the manufacturing of pest...,[SECTOR] Basic Materials [INDUSTRY_GROUP] Agri...
1,101,Basic Materials,10110,Agriculture,10110010,Agricultural Inputs,1011001002,Anhydrous Ammonia Manufacturer,Companies engaged in the manufacturing of anhy...,[SECTOR] Basic Materials [INDUSTRY_GROUP] Agri...
2,101,Basic Materials,10110,Agriculture,10110010,Agricultural Inputs,1011001003,Agricultural Input Distributor,"Companies engaged in the trading, distribution...",[SECTOR] Basic Materials [INDUSTRY_GROUP] Agri...
3,101,Basic Materials,10110,Agriculture,10110010,Agricultural Inputs,1011001004,Fertilizer Manufacturer,Companies engaged in the manufacturing of fert...,[SECTOR] Basic Materials [INDUSTRY_GROUP] Agri...
4,101,Basic Materials,10110,Agriculture,10110010,Agricultural Inputs,1011001005,Nitric Acid Manufacturer,Companies engaged in the manufacturing of nitr...,[SECTOR] Basic Materials [INDUSTRY_GROUP] Agri...


Taxonomy subindustries: 445


## Load Task 2 and Join Task 1 LongProfile

In [57]:
assert TASK2_PATH.exists(), f"Missing {TASK2_PATH}"
assert TASK1_PATH.exists(), f"Missing {TASK1_PATH}"

df = pd.read_csv(TASK2_PATH)
task1 = pd.read_csv(TASK1_PATH, usecols=["CompanyId", "AsOfDate", "LongProfile"])

for frame in [df, task1]:
    frame["CompanyId"] = frame["CompanyId"].astype(str)
    frame["AsOfDate"] = frame["AsOfDate"].astype(str)

df["leaf"] = df["SubIndustry"].map(lambda x: clean_code(x, width=10))
df["parent"] = df["leaf"].str[:8]
df["segment_text"] = [
    build_segment_text(name, desc)
    for name, desc in zip(df["SegmentName"], df["SegmentDescription"])
]

exact_profiles = (
    task1.dropna(subset=["LongProfile"])
    .drop_duplicates(["CompanyId", "AsOfDate"])[["CompanyId", "AsOfDate", "LongProfile"]]
    .rename(columns={"LongProfile": "LongProfile_exact"})
)
latest_profiles = (
    task1.dropna(subset=["LongProfile"])
    .sort_values("AsOfDate")
    .drop_duplicates("CompanyId", keep="last")[["CompanyId", "LongProfile"]]
    .rename(columns={"LongProfile": "LongProfile_latest"})
)

df = df.merge(exact_profiles, on=["CompanyId", "AsOfDate"], how="left")
df = df.merge(latest_profiles, on="CompanyId", how="left")
df["LongProfile"] = df["LongProfile_exact"].fillna(df["LongProfile_latest"])
df["profile_text"] = df["LongProfile"].map(build_profile_text)
df = df.drop(columns=["LongProfile_exact", "LongProfile_latest"])

bad_leaf = df[df["leaf"].str.len() != 10]
assert bad_leaf.empty, f"Found {len(bad_leaf)} rows with non-10-digit SubIndustry"

data_leafs = set(df["leaf"])
taxonomy_leafs = set(taxonomy["subindustry_code"])
coverage = pd.DataFrame({
    "metric": [
        "rows",
        "companies",
        "subindustry_classes",
        "parent_classes",
        "profile_coverage",
        "taxonomy_missing_leaf_classes",
    ],
    "value": [
        len(df),
        df["CompanyId"].nunique(),
        df["leaf"].nunique(),
        df["parent"].nunique(),
        float(df["LongProfile"].notna().mean()),
        len(data_leafs - taxonomy_leafs),
    ],
})
display(coverage)
pd.Series(sorted(data_leafs - taxonomy_leafs), name="missing_leaf").to_csv(OUTPUT_DIR / "taxonomy_missing_leaf_codes.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "data_coverage_summary.csv", index=False)

,metric,value
0,rows,27537.0
1,companies,9352.0
2,subindustry_classes,428.0
3,parent_classes,145.0
4,profile_coverage,1.0
5,taxonomy_missing_leaf_classes,11.0


## Group-Aware Split

In [58]:
def make_group_aware_splits(data, min_eval_support=MIN_EVAL_SUPPORT, n_splits=5):
    data = data.reset_index(drop=True).copy()
    y = data["leaf"].astype(str)
    groups = data["CompanyId"].astype(str)

    support = y.value_counts()
    rare_leafs = set(support[support < min_eval_support].index)
    rare_groups = set(data.loc[y.isin(rare_leafs), "CompanyId"].astype(str))
    fixed_train_mask = groups.isin(rare_groups).to_numpy()
    remaining_idx = np.flatnonzero(~fixed_train_mask)

    folds = np.full(len(data), fill_value=-1, dtype=int)
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for fold, (_, fold_idx) in enumerate(splitter.split(data.iloc[remaining_idx], y.iloc[remaining_idx], groups.iloc[remaining_idx])):
            folds[remaining_idx[fold_idx]] = fold
        for warning in caught:
            print("Split warning:", str(warning.message))

    train_mask = fixed_train_mask | ((folds >= 0) & (~np.isin(folds, [0, 1])))
    val_mask = folds == 1
    test_mask = folds == 0

    for eval_mask in [val_mask, test_mask]:
        missing_eval_leafs = set(y[eval_mask]) - set(y[train_mask])
        if missing_eval_leafs:
            move_groups = set(data.loc[eval_mask & y.isin(missing_eval_leafs).to_numpy(), "CompanyId"].astype(str))
            move_mask = eval_mask & groups.isin(move_groups).to_numpy()
            train_mask |= move_mask
            eval_mask[move_mask] = False

    split = np.full(len(data), "train", dtype=object)
    split[val_mask] = "val"
    split[test_mask] = "test"
    split_df = data[["CompanyId", "leaf", "parent"]].copy()
    split_df["split"] = split
    split_df["fixed_train_rare_group"] = fixed_train_mask
    return split_df

split_df = make_group_aware_splits(df)
df = df.drop(columns=["split", "fixed_train_rare_group"], errors="ignore")
df = df.join(split_df[["split", "fixed_train_rare_group"]])

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

assert set(val_df["leaf"]) <= set(train_df["leaf"]), "Validation contains labels absent from train"
assert set(test_df["leaf"]) <= set(train_df["leaf"]), "Test contains labels absent from train"
assert set(train_df["CompanyId"]).isdisjoint(set(val_df["CompanyId"])), "Train/val company leakage"
assert set(train_df["CompanyId"]).isdisjoint(set(test_df["CompanyId"])), "Train/test company leakage"
assert set(val_df["CompanyId"]).isdisjoint(set(test_df["CompanyId"])), "Val/test company leakage"

split_df.to_csv(OUTPUT_DIR / "split_assignments.csv", index=False)
print(df["split"].value_counts().to_string())
print("Train leaf classes:", train_df["leaf"].nunique())
print("Val leaf classes:", val_df["leaf"].nunique())
print("Test leaf classes:", test_df["leaf"].nunique())

Split warning: The least populated class in y has only 2 members, which is less than n_splits=5.
split
train    16691
test      5425
val       5421
Train leaf classes: 428
Val leaf classes: 368
Test leaf classes: 372


## Ranking Metrics

In [66]:
def topk_recall_from_scores(y_true, scores, classes, ks=(1, 3, 5)):
    classes = np.asarray(classes).astype(str)
    y_true = np.asarray(y_true).astype(str)
    order = np.argsort(-scores, axis=1)
    metrics = {}
    for k in ks:
        pred_topk = classes[order[:, :k]]
        metrics[f"top{k}_recall"] = float(np.mean([yt in row for yt, row in zip(y_true, pred_topk)]))
    return metrics

def ranking_metrics(true_leafs, ranked_leafs, ks=(1, 3, 5)):
    true_leafs = list(map(str, true_leafs))
    metrics = {}
    for k in ks:
        metrics[f"top{k}_accuracy"] = float(np.mean([
            true in list(map(str, preds[:k]))
            for true, preds in zip(true_leafs, ranked_leafs)
        ]))
    reciprocal_ranks = []
    for true, preds in zip(true_leafs, ranked_leafs):
        preds = list(map(str, preds))
        reciprocal_ranks.append(1.0 / (preds.index(true) + 1) if true in preds else 0.0)
    metrics["mrr"] = float(np.mean(reciprocal_ranks))
    return metrics

def print_metrics(title, metrics):
    print(f"\n{title}")
    for key, value in metrics.items():
        if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
            print(f"{key:25s} {float(value):.4f}")
        else:
            print(f"{key:25s} {value}")

## Task 1 Candidate Model: LongProfile -> Parent Industry

In [60]:
parent_encoder = LabelEncoder()
parent_encoder.fit(sorted(df["parent"].unique()))
parent_classes = parent_encoder.classes_.astype(str)

def build_task1_pipeline():
    features = FeatureUnion([
        ("word", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            sublinear_tf=True,
            min_df=2,
            max_df=0.95,
            ngram_range=(1, 2),
            max_features=250_000,
        )),
        ("char", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            sublinear_tf=True,
            analyzer="char_wb",
            min_df=2,
            ngram_range=(3, 5),
            max_features=100_000,
        )),
    ])
    clf = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-5,
        max_iter=25,
        tol=1e-4,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    return Pipeline([("features", features), ("clf", clf)])

def task1_predict_scores(model, frame):
    scores = model.predict_proba(frame["profile_text"].fillna("").tolist())
    model_classes = model.named_steps["clf"].classes_.astype(str)
    full_scores = np.zeros((len(frame), len(parent_classes)), dtype=np.float32)
    class_to_col = {code: idx for idx, code in enumerate(parent_classes)}
    for src_idx, code in enumerate(model_classes):
        full_scores[:, class_to_col[code]] = scores[:, src_idx]
    return full_scores

def save_task1_candidates(split_name, frame, scores):
    order = np.argsort(-scores, axis=1)
    rows = []
    for i, (_, row) in enumerate(frame.iterrows()):
        top_codes = parent_classes[order[i, :FALLBACK_PARENT_TOP_K]].tolist()
        top_scores = scores[i, order[i, :FALLBACK_PARENT_TOP_K]].tolist()
        rows.append({
            "row_index": int(row.name),
            "CompanyId": row["CompanyId"],
            "true_parent": row["parent"],
            "true_leaf": row["leaf"],
            "top_parent_codes": json.dumps(top_codes),
            "top_parent_scores": json.dumps(top_scores),
        })
    out = pd.DataFrame(rows)
    out.to_csv(OUTPUT_DIR / f"task1_parent_candidates_{split_name}.csv", index=False)
    return out

if RUN_TASK1_BASELINE:
    t0 = time.time()
    task1_model = build_task1_pipeline()
    task1_model.fit(train_df["profile_text"].fillna("").tolist(), train_df["parent"].astype(str).tolist())
    joblib.dump(task1_model, TASK1_BASELINE_PATH)

    task1_results = {}
    for split_name, frame in [("val", val_df), ("test", test_df)]:
        scores = task1_predict_scores(task1_model, frame)
        metrics = topk_recall_from_scores(frame["parent"].astype(str), scores, parent_classes, ks=(1, 3, 5, 10))
        task1_results[split_name] = metrics
        save_task1_candidates(split_name, frame, scores)
        print_metrics(f"Task 1 {split_name}", metrics)

    with open(OUTPUT_DIR / "task1_parent_candidate_metrics.json", "w", encoding="utf-8") as f:
        json.dump(task1_results, f, indent=2)
    print("Task 1 seconds:", round(time.time() - t0, 1))
else:
    print("RUN_TASK1_BASELINE=False, skipped Task 1 baseline training.")

RUN_TASK1_BASELINE=False, skipped Task 1 baseline training.


## Candidate Generation From Parent Industries

In [61]:
taxonomy_by_leaf = taxonomy.set_index("subindustry_code").to_dict(orient="index")
leaves_by_parent = taxonomy.groupby("industry_code")["subindustry_code"].apply(list).to_dict()
leaves_by_group = taxonomy.groupby("industry_group_code")["subindustry_code"].apply(list).to_dict()
all_taxonomy_leafs = taxonomy["subindustry_code"].astype(str).tolist()

leaf_to_parent = taxonomy.set_index("subindustry_code")["industry_code"].to_dict()
leaf_to_group = taxonomy.set_index("subindustry_code")["industry_group_code"].to_dict()
taxonomy_text_by_leaf = taxonomy.set_index("subindustry_code")["taxonomy_text"].to_dict()

def candidate_leafs_from_parent_codes(parent_codes, min_candidates=MIN_CANDIDATES):
    candidates = []
    for parent in parent_codes:
        candidates.extend(leaves_by_parent.get(str(parent), []))
    seen = []
    for leaf in candidates:
        if leaf not in seen:
            seen.append(leaf)
    if len(seen) < min_candidates:
        # Add frequent training leaves as a conservative fallback.
        frequent_leafs = train_df["leaf"].value_counts().index.astype(str).tolist()
        for leaf in frequent_leafs:
            if leaf not in seen:
                seen.append(leaf)
            if len(seen) >= min_candidates:
                break
    return seen

def oracle_parent_candidates(row):
    return candidate_leafs_from_parent_codes([row["parent"]], min_candidates=0)

def all_leaf_candidates(row):
    return all_taxonomy_leafs

def load_task1_parent_candidates(split_name):
    path = OUTPUT_DIR / f"task1_parent_candidates_{split_name}.csv"
    if not path.exists():
        return None
    cand = pd.read_csv(path)
    cand["top_parent_codes"] = cand["top_parent_codes"].map(json.loads)
    cand["top_parent_scores"] = cand["top_parent_scores"].map(json.loads)
    return cand

print("Candidate generation ready.")
print("Average leaves per parent:", round(np.mean([len(v) for v in leaves_by_parent.values()]), 2))

Candidate generation ready.
Average leaves per parent: 3.07


## Build Task 2 Segment-Taxonomy Pair Data

In [62]:
label_tfidf = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2), min_df=1)
label_matrix = label_tfidf.fit_transform(taxonomy["taxonomy_text"].fillna("").tolist())
leaf_order = taxonomy["subindustry_code"].astype(str).to_numpy()
leaf_to_order_idx = {leaf: idx for idx, leaf in enumerate(leaf_order)}

def semantic_negative_leafs(segment_text, true_leaf, n=NEG_SEMANTIC):
    q = label_tfidf.transform([segment_text])
    sims = (q @ label_matrix.T).toarray().ravel()
    order = np.argsort(-sims)
    negatives = []
    for idx in order:
        leaf = str(leaf_order[idx])
        if leaf != true_leaf and leaf not in negatives:
            negatives.append(leaf)
        if len(negatives) >= n:
            break
    return negatives

def sample_from_pool(pool, true_leaf, n, rng, existing):
    pool = [str(x) for x in pool if str(x) != true_leaf and str(x) not in existing]
    if not pool or n <= 0:
        return []
    rng.shuffle(pool)
    return pool[:n]

def sample_negative_leafs(row, rng):
    true_leaf = str(row["leaf"])
    true_parent = str(row["parent"])
    true_group = leaf_to_group.get(true_leaf, "")
    negatives = []

    negatives += sample_from_pool(leaves_by_parent.get(true_parent, []), true_leaf, NEG_SAME_PARENT, rng, negatives)
    negatives += sample_from_pool(leaves_by_group.get(true_group, []), true_leaf, NEG_SAME_GROUP, rng, negatives)
    for leaf in semantic_negative_leafs(row["segment_text"], true_leaf, NEG_SEMANTIC):
        if leaf not in negatives:
            negatives.append(leaf)
    negatives += sample_from_pool(all_taxonomy_leafs, true_leaf, NEG_RANDOM, rng, negatives)
    return negatives

def build_pair_text(segment_text, taxonomy_text):
    return (
        f"[SEGMENT] {normalize_text(segment_text)} "
        f"[CANDIDATE_TAXONOMY] {normalize_text(taxonomy_text)} "
        "[QUESTION] Does this segment belong to this subindustry?"
    ).strip()

def build_pair_dataframe(frame, split_name, max_rows=DEBUG_PAIR_ROWS):
    rng = random.Random(SEED)
    work = frame.copy()
    if max_rows is not None:
        work = work.sample(n=min(max_rows, len(work)), random_state=SEED).reset_index(drop=True)

    rows = []
    for row_id, row in tqdm(work.iterrows(), total=len(work), desc=f"pairs {split_name}"):
        true_leaf = str(row["leaf"])
        candidate_leafs = [true_leaf] + sample_negative_leafs(row, rng)
        for leaf in candidate_leafs:
            tax_text = taxonomy_text_by_leaf.get(leaf, f"[SUBINDUSTRY_CODE] {leaf} [INDUSTRY_CODE] {leaf[:8]}")
            rows.append({
                "source_row_id": int(row_id),
                "split": split_name,
                "CompanyId": row["CompanyId"],
                "true_leaf": true_leaf,
                "candidate_leaf": leaf,
                "candidate_parent": leaf[:8],
                "label": int(leaf == true_leaf),
                "segment_text": row["segment_text"],
                "taxonomy_text": tax_text,
                "pair_text": build_pair_text(row["segment_text"], tax_text),
            })
    pair_df = pd.DataFrame(rows)
    return pair_df

if RUN_BUILD_PAIR_DATA:
    train_pairs = build_pair_dataframe(train_df, "train")
    val_pairs = build_pair_dataframe(val_df, "val")
    test_pairs = build_pair_dataframe(test_df, "test")
    train_pairs.to_csv(PAIR_TRAIN_PATH, index=False)
    val_pairs.to_csv(PAIR_VAL_PATH, index=False)
    test_pairs.to_csv(PAIR_TEST_PATH, index=False)
    print("Saved pair data:")
    print(PAIR_TRAIN_PATH, len(train_pairs))
    print(PAIR_VAL_PATH, len(val_pairs))
    print(PAIR_TEST_PATH, len(test_pairs))
else:
    print("RUN_BUILD_PAIR_DATA=False, skipped pair-data generation.")

RUN_BUILD_PAIR_DATA=False, skipped pair-data generation.


## Cross-Encoder Dataset and Model

In [63]:
class PairDataset(Dataset):
    def __init__(self, pair_df, tokenizer, max_length=MAX_LENGTH):
        self.texts = pair_df["pair_text"].astype(str).tolist()
        self.labels = pair_df["label"].astype(np.float32).to_numpy()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

def masked_mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class SegmentTaxonomyCrossEncoder(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, local_files_only=LOCAL_FILES_ONLY)
        if FORCE_ENCODER_FLOAT32:
            self.encoder.float()
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        if getattr(outputs, "pooler_output", None) is not None:
            pooled = outputs.pooler_output
        else:
            pooled = masked_mean_pool(outputs.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        pooled = pooled.to(self.classifier.weight.dtype)
        return self.classifier(pooled).squeeze(-1)

if RUN_SMOKE_FORWARD:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=LOCAL_FILES_ONLY)
    sample_pair = pd.DataFrame({
        "pair_text": [
            build_pair_text(train_df.iloc[0]["segment_text"], taxonomy_text_by_leaf.get(train_df.iloc[0]["leaf"], "")),
            build_pair_text(train_df.iloc[1]["segment_text"], taxonomy_text_by_leaf.get(train_df.iloc[1]["leaf"], "")),
        ],
        "label": [1, 1],
    })
    ds = PairDataset(sample_pair, tok)
    batch = next(iter(DataLoader(ds, batch_size=2)))
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    model = SegmentTaxonomyCrossEncoder().to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    print("Smoke logits shape:", tuple(logits.shape), "dtype:", logits.dtype)
    del model, tok, ds, batch
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_FORWARD=False, skipped cross-encoder smoke forward.")

RUN_SMOKE_FORWARD=False, skipped cross-encoder smoke forward.


## Train Cross-Encoder Reranker

In [64]:
@torch.no_grad()
def predict_pair_scores(model, pair_df, tokenizer, batch_size=EVAL_BATCH_SIZE):
    ds = PairDataset(pair_df, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    scores = []
    labels = []
    model.eval()
    for batch in tqdm(loader, desc="score pairs", leave=False):
        labels.append(batch["labels"].numpy())
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        scores.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(scores), np.concatenate(labels)

def pair_auc(scores, labels):
    if len(np.unique(labels)) < 2:
        return float("nan")
    return float(roc_auc_score(labels, scores))

def train_cross_encoder():
    assert PAIR_TRAIN_PATH.exists(), f"Missing {PAIR_TRAIN_PATH}. Run pair generation first."
    assert PAIR_VAL_PATH.exists(), f"Missing {PAIR_VAL_PATH}. Run pair generation first."

    train_pairs = pd.read_csv(PAIR_TRAIN_PATH)
    val_pairs = pd.read_csv(PAIR_VAL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=LOCAL_FILES_ONLY)
    train_ds = PairDataset(train_pairs, tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    model = SegmentTaxonomyCrossEncoder().to(DEVICE)
    pos = train_pairs["label"].sum()
    neg = len(train_pairs) - pos
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_auc = -1.0
    patience = 0
    history = []
    model_dir = OUTPUT_DIR / "cross_encoder"
    model_dir.mkdir(parents=True, exist_ok=True)
    best_path = model_dir / "best_state.pt"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        start = time.time()
        for batch in tqdm(train_loader, desc=f"cross-encoder epoch {epoch}", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            loss = loss_fn(logits, batch["labels"])
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss detected: {loss.item()}")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))

        val_scores, val_labels = predict_pair_scores(model, val_pairs, tokenizer)
        val_auc = pair_auc(val_scores, val_labels)
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "val_pair_auc": val_auc,
            "seconds": round(time.time() - start, 1),
        }
        history.append(row)
        print(row)
        pd.DataFrame(history).to_csv(model_dir / "history.csv", index=False)
        if val_auc > best_auc:
            best_auc = val_auc
            patience = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience += 1
            if patience >= EARLY_STOP_PATIENCE:
                print("Early stopping.")
                break

    with open(model_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump({"best_val_pair_auc": best_auc}, f, indent=2)
    return model_dir

if RUN_CROSS_ENCODER_TRAINING:
    cross_encoder_dir = train_cross_encoder()
    print("Saved cross-encoder to:", cross_encoder_dir)
else:
    print("RUN_CROSS_ENCODER_TRAINING=False, skipped cross-encoder training.")

RUN_CROSS_ENCODER_TRAINING=False, skipped cross-encoder training.


## Full Candidate Ranking Evaluation

In [67]:
def candidate_rows_for_frame(frame, candidate_mode="oracle_parent", split_name="val"):
    task1_candidates = load_task1_parent_candidates(split_name)
    task1_by_pos = {}
    if task1_candidates is not None:
        task1_by_pos = {i: row for i, row in task1_candidates.iterrows()}

    rows = []
    for pos, row in frame.reset_index(drop=True).iterrows():
        if candidate_mode == "oracle_parent":
            candidate_leafs = oracle_parent_candidates(row)
        elif candidate_mode == "task1_topk":
            assert task1_candidates is not None, f"Missing Task 1 candidates for {split_name}"
            parent_codes = task1_by_pos[pos]["top_parent_codes"][:TASK1_TOP_K]
            parent_scores = task1_by_pos[pos]["top_parent_scores"][:TASK1_TOP_K]
            if parent_scores and parent_scores[0] < LOW_CONFIDENCE_THRESHOLD:
                parent_codes = task1_by_pos[pos]["top_parent_codes"][:FALLBACK_PARENT_TOP_K]
            candidate_leafs = candidate_leafs_from_parent_codes(parent_codes)
        elif candidate_mode == "all_taxonomy":
            candidate_leafs = all_leaf_candidates(row)
        else:
            raise ValueError(candidate_mode)

        if row["leaf"] not in candidate_leafs and candidate_mode != "task1_topk":
            candidate_leafs = [row["leaf"]] + candidate_leafs
        for leaf in candidate_leafs:
            tax_text = taxonomy_text_by_leaf.get(leaf, f"[SUBINDUSTRY_CODE] {leaf} [INDUSTRY_CODE] {leaf[:8]}")
            rows.append({
                "source_row_id": pos,
                "true_leaf": row["leaf"],
                "candidate_leaf": leaf,
                "pair_text": build_pair_text(row["segment_text"], tax_text),
                "label": int(leaf == row["leaf"]),
            })
    return pd.DataFrame(rows)

def evaluate_cross_encoder_ranking(model_dir, frame, split_name="val", candidate_mode="oracle_parent"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=LOCAL_FILES_ONLY)
    model = SegmentTaxonomyCrossEncoder().to(DEVICE)
    model.load_state_dict(torch.load(Path(model_dir) / "best_state.pt", map_location=DEVICE))
    pair_df = candidate_rows_for_frame(frame, candidate_mode=candidate_mode, split_name=split_name)
    pair_df["score"], _ = predict_pair_scores(model, pair_df.assign(label=0), tokenizer)

    ranked = []
    true_leafs = []
    for row_id, group in pair_df.groupby("source_row_id", sort=True):
        group = group.sort_values("score", ascending=False)
        ranked.append(group["candidate_leaf"].astype(str).tolist())
        true_leafs.append(group["true_leaf"].iloc[0])
    metrics = ranking_metrics(true_leafs, ranked, ks=(1, 3, 5))
    metrics["candidate_mode"] = candidate_mode
    metrics["split"] = split_name
    metrics["avg_candidates"] = float(pair_df.groupby("source_row_id").size().mean())
    return metrics, pair_df

if RUN_FULL_PIPELINE_EVAL:
    model_dir = OUTPUT_DIR / "cross_encoder"
    assert (model_dir / "best_state.pt").exists(), f"Missing cross-encoder model: {model_dir / 'best_state.pt'}"
    all_metrics = []
    for candidate_mode in ["oracle_parent", "task1_topk"]:
        for split_name, frame in [("val", val_df), ("test", test_df)]:
            metrics, scored_pairs = evaluate_cross_encoder_ranking(model_dir, frame, split_name=split_name, candidate_mode=candidate_mode)
            all_metrics.append(metrics)
            print_metrics(f"{split_name} {candidate_mode}", metrics)
            scored_pairs.to_csv(OUTPUT_DIR / f"scored_pairs_{split_name}_{candidate_mode}.csv", index=False)
    pd.DataFrame(all_metrics).to_csv(OUTPUT_DIR / "full_pipeline_ranking_metrics.csv", index=False)
else:
    print("RUN_FULL_PIPELINE_EVAL=False, skipped full ranking evaluation.")

[transformers] The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 22259.05it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.


val oracle_parent
top1_accuracy             0.8607
top3_accuracy             0.9867
top5_accuracy             0.9965
mrr                       0.9229
candidate_mode            oracle_parent
split                     val
avg_candidates            3.9076


[transformers] The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 18454.00it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.


test oracle_parent
top1_accuracy             0.8575
top3_accuracy             0.9888
top5_accuracy             0.9983
mrr                       0.9214
candidate_mode            oracle_parent
split                     test
avg_candidates            3.9097


[transformers] The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 17649.30it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.


val task1_topk
top1_accuracy             0.3992
top3_accuracy             0.6514
top5_accuracy             0.7364
mrr                       0.5426
candidate_mode            task1_topk
split                     val
avg_candidates            34.3540


[transformers] The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 19683.41it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.


test task1_topk
top1_accuracy             0.3840
top3_accuracy             0.6533
top5_accuracy             0.7397
mrr                       0.5365
candidate_mode            task1_topk
split                     test
avg_candidates            34.3176


## How to Use This Notebook

First run with all flags set to `False` to confirm setup.

Then run in this order:

```python
RUN_TASK1_BASELINE = True
RUN_BUILD_PAIR_DATA = False
RUN_CROSS_ENCODER_TRAINING = False
RUN_FULL_PIPELINE_EVAL = False
```

Then:

```python
RUN_TASK1_BASELINE = False
RUN_BUILD_PAIR_DATA = True
RUN_CROSS_ENCODER_TRAINING = False
RUN_FULL_PIPELINE_EVAL = False
```

Then:

```python
RUN_TASK1_BASELINE = False
RUN_BUILD_PAIR_DATA = False
RUN_CROSS_ENCODER_TRAINING = True
RUN_FULL_PIPELINE_EVAL = False
```

Finally:

```python
RUN_TASK1_BASELINE = False
RUN_BUILD_PAIR_DATA = False
RUN_CROSS_ENCODER_TRAINING = False
RUN_FULL_PIPELINE_EVAL = True
```

Use `DEBUG_PAIR_ROWS = 1000` for a quick cross-encoder dry run before building full pair data.